# Analytical SQL Queries over the Trossen Catalog

Cross-episode questions answered with **DataFusion SQL** (`client.ctx.sql`). The
polished versions live in `trossen_oss.query`; here we also show the raw
`register_view` + `sql` pattern so you can adapt it. Start `pixi run serve` and
`pixi run register` first.

In [ ]:
from __future__ import annotations

import os

import rerun as rr
from rerun.notebook import Viewer

from trossen_oss.catalog import DEFAULT_CATALOG_URL

SERVER = os.getenv("RERUN_CLOUD_ADDR", DEFAULT_CATALOG_URL)
DATASET_NAME = os.getenv("DATASET_NAME", "trossen_oss")

client = rr.catalog.CatalogClient(SERVER)
dataset = client.get_dataset(DATASET_NAME)
ctx = client.ctx

## Schema for the Trossen episodes

- **timeline**: `message_log_time`
- **per-joint scalars**: `/robot_{left,right}/joints/joint_{0..5}` → component
  `Scalars:scalars` (a one-element list — index it `[1]` in SQL)

In [ ]:
TIMELINE = "message_log_time"
RIGHT_JOINT = "/robot_right/joints/joint_1"

## Dataset inventory (raw SQL)

The `register_view` + `sql` pattern over the cheap `segment_table()`.

In [ ]:
ctx.register_view("segments", dataset.segment_table())
ctx.sql(
    "SELECT rerun_segment_id, rerun_layer_names, rerun_num_chunks, rerun_size_bytes"
    " FROM segments ORDER BY rerun_segment_id LIMIT 20"
).to_arrow_table()

## Per-arm motion — which arm is scripted vs. task-driven

`trossen_oss.query.arm_activity` — one cross-segment reader over all 12 joints,
`SUM(MAX − MIN)` per arm, grouped by `rerun_segment_id`.

In [ ]:
from trossen_oss.query import arm_activity

arm_activity(client, dataset)

## Velocity-limit violations (window functions)

`trossen_oss.query.velocity_limit_violations` — a `LAG` window over the per-segment
time order gives velocity; we count samples above the `π rad/s` limit and the peak.

In [ ]:
from trossen_oss.query import velocity_limit_violations

velocity_limit_violations(client, dataset, RIGHT_JOINT)

## Inspect a flagged episode in the Viewer

Pick the worst episode above, then browse to it in the embedded catalog Viewer.

In [ ]:
Viewer(url=SERVER, width="auto", height=500)